# VaR / ES model comparison: walkthrough
Eight models, rolling 500-day window, back-tests and consistent ranking. All logic lives in the `marketrisk` package.

In [1]:
SOURCE = "yahoo"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from marketrisk import data as D
pd.set_option('display.precision', 3)
R = (D.to_returns(D.load_prices(cache_dir='../data/raw')) if SOURCE == 'yahoo'
     else D.make_synthetic(3000))
w = D.equal_weights(R.shape[1])
R.describe().T[['mean', 'std', 'min', 'max']].mul(100).round(2)

,mean,std,min,max
Ticker,,,,
SPY,0.05,1.20,-10.94,14.52
QQQ,0.06,1.36,-11.98,12.16
IWM,0.04,1.52,-13.27,9.15
EFA,0.03,1.33,-11.16,15.89
EEM,0.04,1.74,-16.17,22.77
VNQ,0.04,1.81,-19.51,17.01
TLT,0.02,0.92,-6.67,7.52
IEF,0.01,0.43,-2.51,3.43
LQD,0.02,0.54,-9.11,9.77


## Stylised facts: fat tails and volatility clustering

In [2]:
from scipy import stats
r = D.portfolio_returns(R, w)
print(f'excess kurtosis {stats.kurtosis(r):.1f}, skew {stats.skew(r):.2f}')
print('autocorr of r  :', round(r.autocorr(), 3))
print('autocorr of r^2:', round((r**2).autocorr(), 3))

excess kurtosis 12.6, skew -0.02
autocorr of r  : -0.088
autocorr of r^2: 0.278


## Rolling out-of-sample forecasts

In [3]:
from marketrisk.backtest import run_rolling, summary_table, period_exceptions, STRESS_PERIODS
from marketrisk.models import default_models
res = run_rolling(R, w, default_models(), window=500)
summary_table(res)

,exceptions,expected,rate_%,kupiec_p,christ_ind_p,christ_cc_p,days_red_%,max_exc_250d,es_z2,es_zone,avg_var_%
model,,,,,,,,,,,
Historical,75,47.8,1.569,2.634e-04,1.585e-04,1.024e-06,12.820,21,0.402,green,2.084
Normal,108,47.8,2.259,5.894e-14,9.590e-08,3.738e-19,21.447,27,0.848,yellow,1.783
Student-t,79,47.8,1.652,3.408e-05,4.125e-05,4.163e-08,12.555,23,0.572,green,1.970
Cornish-Fisher,44,47.8,0.920,5.746e-01,3.834e-05,1.781e-04,5.075,13,-0.014,green,2.700
EWMA,99,47.8,2.071,7.842e-11,5.194e-03,1.316e-11,5.318,13,0.617,green,1.661
GARCH-t,79,47.8,1.652,3.408e-05,2.059e-03,1.616e-06,2.670,11,0.540,green,1.742
FHS,67,47.8,1.401,8.537e-03,2.536e-03,3.302e-04,0.000,8,0.160,green,1.852
Monte Carlo-t,74,47.8,1.548,4.257e-04,1.325e-04,1.356e-06,5.274,12,0.386,green,1.825


In [4]:
period_exceptions(res, STRESS_PERIODS)

,GFC 2008 (146d),Euro crisis 2011 (127d),COVID 2020 (72d),Rates 2022 (251d),full (4781d)
model,,,,,
Historical,10.959,3.150,12.500,1.992,1.569
Normal,15.753,4.724,12.500,3.984,2.259
Student-t,13.699,3.150,12.500,2.390,1.652
Cornish-Fisher,7.534,3.150,9.722,1.195,0.920
EWMA,2.740,2.362,9.722,2.390,2.071
GARCH-t,2.740,3.150,6.944,1.992,1.652
FHS,2.055,3.150,5.556,1.195,1.401
Monte Carlo-t,2.740,2.362,9.722,2.390,1.548


## Ranking with consistent scoring functions
FZ0 is the joint (VaR, ES) loss; Diebold-Mariano compares each model with the best.

In [5]:
from marketrisk.compare import loss_table, ranking
ranking(loss_table(res))

,rank,quantile_loss_x1e4,fz0_loss,dm_vs_best,dm_p_value
model,,,,,
FHS,1,2.544,-3.975,NaN,NaN
Monte Carlo-t,2,2.615,-3.941,1.456,1.454e-01
GARCH-t,3,2.572,-3.937,2.609,9.081e-03
EWMA,4,2.688,-3.914,2.298,2.156e-02
Cornish-Fisher,5,3.487,-3.726,5.269,1.369e-07
Historical,6,3.499,-3.712,3.144,1.665e-03
Student-t,7,3.509,-3.672,3.131,1.742e-03
Normal,8,3.635,-3.618,3.088,2.018e-03


## Custom portfolio: 60/40 equities/Treasuries

In [6]:
w6040 = np.zeros(R.shape[1]); cols = list(R.columns)
w6040[cols.index('SPY')], w6040[cols.index('TLT')] = 0.6, 0.4
res6040 = run_rolling(R, w6040, default_models(mc_sims=5000), window=500)
summary_table(res6040)[['exceptions', 'expected', 'kupiec_p', 'christ_cc_p', 'days_red_%']]

,exceptions,expected,kupiec_p,christ_cc_p,days_red_%
model,,,,,
Historical,71,47.8,1.665e-03,2.093e-10,5.494
Normal,113,47.8,7.814e-16,4.783e-26,19.682
Student-t,87,47.8,3.218e-07,5.219e-13,13.416
Cornish-Fisher,47,47.8,9.060e-01,3.954e-04,0.309
EWMA,93,47.8,6.075e-09,4.649e-10,1.037
GARCH-t,76,47.8,1.609e-04,9.876e-05,1.015
FHS,57,47.8,1.948e-01,8.143e-03,0.000
Monte Carlo-t,70,47.8,2.555e-03,2.625e-05,0.816
